# CS336 Assignment 1 (basics): Building a Transformer LM


## Problem (unicode1): Understanding Unicode (1 point)


#### (a) What Unicode character does chr(0) return?


`chr(0)` returns the Unicode character U+0000, which is known as the null character (often abbreviated as NUL).

Originally, it used the end of string in C-style strings.

It's non-printable control character, and typically has no visible representation.


In [19]:
chr(0)
print(chr(0))

 


#### (b) How does this character’s string representation (**repr**()) differ from its printed representation?


`__repr__() ` method returns representation of the character or object which internally used in Python.

On the other hand, `__str__()` method returns visual reprentation of the character or object which human can read.


#### (c) What happens when this character occurs in text?


In [20]:
chr(0)
print(chr(0))
"this is a test" + chr(0) + "string"
print("this is a test" + chr(0) + "string")

 
this is a test string


In C style strings, `chr(0)` is used to terminate the string.

However in Python, `chr(0)` is treated as a regular character, NUL.


## Problem (unicode2): Unicode Encodings (3 points)


#### (a) What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than UTF-16 or UTF-32?


UTF-8 can treat ASCII in one byte. However, UTF-16 treats ASCII 2 bytes and UTF-32 treats ASCII 4 bytes. There is a space efficiency.


In [21]:
print(len("a".encode("utf-8")))
print(len("a".encode("utf-16le")))
print(len("a".encode("utf-16")))
print(len("a".encode("utf-32le")))
print(len("a".encode("utf-32")))

1
2
4
4
8


#### (b) Consider the following (incorrect) function, which is intended to decode a UTF-8 byte string into a Unicode string. Why is this function incorrect?


In [22]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])


decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))

'hello'

위 코드는 ASCII 집합에 들어가는 것만 제대로 표현할 수 있고, 아래와 같이 한글이나 한자 등 ASCII 집합에 속하지 않는 문자열은 표현할 수 없습니다.


In [23]:
decode_utf8_bytes_to_str_wrong("안녕하세요!".encode("utf-8"))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xec in position 0: unexpected end of data

#### (c) Give a two byte sequence that does not decode to any Unicode character(s).


UTF-8은 가변 길이 인코딩이기 때문에 각 바이트의 비트 패턴에 따라 몇 바이트로 해석할지 규칙이 필요합니다. 이 규칙을 어긴 바이트 시퀀스는 디코딩 시 UnicodeDecodeError가 발생합니다. 예를 들어, overlong encoding이나 잘못된 continuation byte 등이 해당됩니다.

반면 UTF-32는 고정 길이 4바이트 인코딩이므로 바이트 단위로 나눌 필요 없이 4바이트씩 끊어서 바로 디코딩이 가능합니다. 그러나 모든 4바이트 조합이 유효한 유니코드 문자로 해석되는 것은 아닙니다. 예를 들어, surrogate 범위(U+D800–U+DFFF)나 유효 범위를 벗어난 값은 UTF-32에서도 UnicodeDecodeError를 일으킬 수 있습니다.


아래 표와 같이 첫번째 바이트 값에 따라 어디까지가 한 문자인지 알 수 있습니다.

| 첫 바이트 | 문자 크기    |
| --------- | ------------ |
| 0 ~ 127   | 1 byte       |
| 128 ~ 191 | x            |
| 192 ~ 193 | x (Overlong) |
| 194 ~ 223 | 2 bytes      |
| 224 ~ 239 | 3 bytes      |
| 240 ~ 247 | 4 bytes      |
| 248 ~ 255 | x            |


In [10]:
b"\xc1\x00".decode("utf-8")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc1 in position 0: invalid start byte

## Problem (train_bpe): BPE Tokenizer Training (15 points)
